# Comparing distribution of germline and somatic events by molecular consequence

## cBioPortal vs ClinVar:

Search for the term 'USER' to locate codes lines where user must provide input

In [175]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
#initialize dfs to collect info gene-wise
#1-VEP categories that are filtered out (expected to be filtered by cbio so removing from analysis, but outputting to this df for counts and QC)
VEP_categories_filtered_45TSGs=pd.DataFrame()
#2-all TSG, including splice region, all output stats to this df gene-wise
allTSG_df=pd.DataFrame()
#3-all TSG, excluding splice region, all output stats to this df gene-wise
allTSG_df_nospliceregion=pd.DataFrame()

FLAG1_count=FLAG5_count=FLAGFISHER_count=0

parentpath="USER INSERT PARENT PATH TO WHERE INDIVIDUAL GENE FOLDERS ARE STORED"
os.chdir(parentpath)
# loading VEP consequence titles (category names) to use as index to join cbio and clinvar processed dfs:
VEP_calc_consequences=pd.read_csv("USER INPUT FILE WITH LIST OF POSSIBLE CONSEQUENCES FROM ENSEMBL VEP", sep="\t", header=None)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE TRANSCRIPT ID (Supplemental Table S4)", sep="\t")

Hypermutators=pd.read_csv("USER INPUT FILE WITH SAMPLE IDS IN CBIOPORTAL HAVING TUMOR MUTATION BURDEN GREATER THAN 10", sep="\t")


for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name to run separate comparison tests per gene
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        clinvar_cbio_tocompare=pd.read_csv("USER INPUT FILE NAME OF SOMATIC VARIANTS IN CLINVAR UNFILTERED FOR PATHOGENICITY", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        print("shared count")
        print(len(clinvar_cbio_tocompare))
        print("shared that are VUSLBB")
        print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("USER INPUT PROCESSED SOMATIC DATA FILE NAME", sep="\t")
        print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        print("cbio VEP len after filter VEP categories")
        print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        print("with hypermutators count QC same as above")
        totalbeforehypermutatorfilter=len(cbio_VEP)
        print(len(cbio_VEP))
        cbio_VEP_nohypermutator=cbio_VEP[~cbio_VEP["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_VEP=cbio_VEP_nohypermutator
        print("without hypermutators count")
        print(len(cbio_VEP))
        print("% vars lost to hypermutator filter=#total-#afterfilter/#total")
        print(((totalbeforehypermutatorfilter-len(cbio_VEP))*100)/totalbeforehypermutatorfilter)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        print("without VUSLBB")
        print(len(cbio_VEP_noVUSLBB))
        print("# variants lost after VUSLBB filter")
        print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertype=cbio_VEP_noVUSLBB
        #initialize default dict to select 1 variant at random from list of variants per patient (for patients with multiple variants in the same gene):
        cbio_VEP_dict=defaultdict(list)
        # loading VEP consequence titles (categories) to use as index to join randomavg dfs:
        cbio_VEP_calc_consequences=VEP_calc_consequences
        cbio_VEP_randomavg_means=cbio_VEP_calc_consequences.set_index(0)
        #making dictionary of list of variants per patient id in the form ({patient1: [var1consequence, var2consequence, var3consequence,...], patient2:[var1consequence,var2consequence,..]})
        for index, row in cbio_VEP_cancertype.iterrows():
            patientid=cbio_VEP_cancertype.loc[index]["patientId"]
            VEPresult=cbio_VEP_cancertype.loc[index]["collapsed_Consequence"]
            cbio_VEP_dict[patientid].append(VEPresult)

        #randomly selecting 1 variant per patient id and summarizing distribution, repeating 5 times
        for i in range(5):
            cbio_VEP_random = {k:random.choice(v) for k,v in list(cbio_VEP_dict.items())}
            list_VEP_random=[]
            for key, val in cbio_VEP_random.items():
                eachrow=[key,val]
                list_VEP_random.append(eachrow)
            #convert list to df of 1 variant per patient [patient1:var2consequence, patient2:var1consequence, ...]
            dfconvert= pd.DataFrame(list_VEP_random)
            #summarize counts per VEP category and convert to df having VEP category as index
            cbio_VEP_randomavg=pd.DataFrame(dfconvert[1].value_counts())
            #above is a df of current random selection having row header=VEP category and only 1 column of the counts per category (aka row) derived from current random selection
            #rename counts column in above df with the loop number (1/2/3/4/5) + join together with previous df (if not the 1st random selection of 5) and calc mean of all 5 sets of distributions
            rename=f"VEP_count_{i}"
            cbio_VEP_randomavg_rename=cbio_VEP_randomavg.rename(columns={1:rename})
            cbio_VEP_randomavg_means=cbio_VEP_randomavg_means.join(cbio_VEP_randomavg_rename)
        #replace all missing values with 0 to calc mean
        cbio_VEP_randomavg_means_fill0=cbio_VEP_randomavg_means.fillna(0)
        #calc mean
        cbio_VEP_randomavg_means_fill0['mean']=cbio_VEP_randomavg_means_fill0.mean(axis=1)
        
        clinvar_VEP_cancertype=pd.read_csv("USER INPUT PROCESSED GERMLINE DATA FILE NAME", sep="\t")
        
        clinvar_VEP_MANE=VEP_calc_consequences.set_index(0)
        #create list with each VEP category in data, plus occurrence list of number of times each variant seen in that category
        clinvar_VEPresult=defaultdict(list)
        for index, row in clinvar_VEP_cancertype.iterrows():
            VEPresult=clinvar_VEP_cancertype.loc[index]["collapsed_Consequence"]
            variantcount=clinvar_VEP_cancertype.loc[index]["Occurrence"]
            clinvar_VEPresult[VEPresult].append(variantcount)
        #sum the list of occurrence per VEP category
        clinvar_VEPresult_sumoccurrence = {k: sum(v) for (k, v) in clinvar_VEPresult.items()}
        #create dataframe from dictionary
        clinvar_dfconvert=pd.DataFrame.from_dict(clinvar_VEPresult_sumoccurrence, orient='index')
        #join with VEP consequence df
        clinvar_VEP=clinvar_VEP_MANE.join(clinvar_dfconvert)
        #replace missing values with 0
        clinvar_VEP_fill0=clinvar_VEP.fillna(0)
        #rename germline column
        clinvar_VEP_fill0_rename=clinvar_VEP_fill0.rename(columns={0:"VEP_germline"})
        #rename somatic column in cbio df
        cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0.rename(columns={"mean":"VEP_somatic"})
        #combine gl and s dfs
        df_gl_s = clinvar_VEP_fill0_rename.join(cbio_VEP_randomavg_means_fill0_rename)
        #filter out random selection columns
        df_gl_s=df_gl_s.filter(items=["VEP_germline","VEP_somatic"])
        #filter out categories filtered by cbio (excep for TERT- diff set since cbio includes promoter variants)
        if folder_name=="TERT":
            VEP_categories_tofilter_list=["synonymous_variant", "3_prime_UTR_variant", "intron_variant", "downstream_gene_variant"]
        else:    
            VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant"]

        df_gl_s_filter=df_gl_s.drop(VEP_categories_tofilter_list)
        
        #create df of VEP cat filtered out for all 45 TSGs (to QC)
        VEP_categories_filtered=df_gl_s.drop(df_gl_s_filter.index)
        #VEP_categories_filtered.to_csv("Python_coderun_8-24-23/Python_VEP_categories_filtered_8-29-23.txt", sep="\t")
        Gene= pd.DataFrame({0: [folder_name]})
        Germline= pd.DataFrame({0: ["Germline"]})
        Somatic= pd.DataFrame({0: ["Somatic"]})
        VEP_categories_filtered_output=pd.concat([Gene, Germline, VEP_categories_filtered["VEP_germline"], Somatic, VEP_categories_filtered["VEP_somatic"]])
        VEP_categories_filtered_45TSGs=pd.concat([VEP_categories_filtered_45TSGs,VEP_categories_filtered_output])
        
        df_gl_s_allgte5 = df_gl_s_filter[(df_gl_s_filter["VEP_somatic"] >0)| (df_gl_s_filter["VEP_germline"] >0)]
        
        
        #filter out splice region for sep chi square stats
        if "splice_region_variant" in df_gl_s_allgte5.index:
            df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5.drop("splice_region_variant")
        else:
            df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5
        
        
        
        #REPEAT WITHOUT SPLICE REGION VARIANTS
        #get total counts of gl and s variants analyzed
        #gl and s total counts
        somatic_counts_nospliceregion=df_gl_s_allgte5_nospliceregion['VEP_somatic'].sum()
        germline_counts_nospliceregion=df_gl_s_allgte5_nospliceregion['VEP_germline'].sum()
        totalcategories=len(df_gl_s_allgte5_nospliceregion)
        print(df_gl_s_allgte5_nospliceregion)
        for index, row in df_gl_s_allgte5_nospliceregion.iterrows():
            if (df_gl_s_allgte5_nospliceregion.loc[index,"VEP_germline"]<(0.01*germline_counts_nospliceregion))&(df_gl_s_allgte5_nospliceregion.loc[index,"VEP_somatic"]<(0.01*somatic_counts_nospliceregion)):
                df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5_nospliceregion.drop(index)
        
        #run chi-square test from scipy
        from scipy.stats import chi2_contingency
        c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5_nospliceregion)
        print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
        print(df_gl_s_allgte5_nospliceregion)
        print(expected_nospliceregion)

        expected_df=pd.DataFrame(expected_nospliceregion)

        #codeblock for rule exp<5 in both gl and s:
        df_gl_s_allgte5_nospliceregion_resetindex=df_gl_s_allgte5_nospliceregion.reset_index()

        indextodrop=[]
        for index, row in expected_df.iterrows():
            if ((expected_df.loc[index,0]<5)&(expected_df.loc[index,1]<5)):
                indextodrop.append(index)
        df_gl_s_allgte5_nospliceregion_resetindex_filter=df_gl_s_allgte5_nospliceregion_resetindex.drop(indextodrop)
        expected_df=expected_df.drop(indextodrop)
        
        df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5_nospliceregion_resetindex_filter.set_index(0)
        
        c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5_nospliceregion)
        print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
        print(df_gl_s_allgte5_nospliceregion)
        print(expected_nospliceregion)

        expected_df=pd.DataFrame(expected_nospliceregion)
        
        somatic_counts_nospliceregion_1=df_gl_s_allgte5_nospliceregion['VEP_somatic'].sum()
        germline_counts_nospliceregion_1=df_gl_s_allgte5_nospliceregion['VEP_germline'].sum()
        
        
        #FLAGS:
        FLAG1=FLAG5=FLAGFISHER=0

        for index, row in expected_df.iterrows():
            if (expected_df.loc[index,0]<1)|(expected_df.loc[index,1]<1):
                FLAG1=1
                FLAG1_count=FLAG1_count+1
       
        
        lt5cellcount=pd.DataFrame(expected_df[expected_df<5].count())
        lt5cellcount=lt5cellcount[0].sum()
        if lt5cellcount>(0.2*expected_df.size):
            FLAG5=1
            FLAG5_count=FLAG5_count+1
        
        for index, row in df_gl_s_allgte5_nospliceregion.iterrows():
            if (df_gl_s_allgte5_nospliceregion.loc[index,"VEP_germline"]<1)|(df_gl_s_allgte5_nospliceregion.loc[index,"VEP_somatic"]<1):
                FLAGFISHER=1
                FLAGFISHER_count=FLAGFISHER_count+1
        
        print("FLAG1")
        print(FLAG1)
        print("FLAG5")
        print(FLAG5)
        print("FLAGFISHER")
        print(FLAGFISHER)
        
        #get Pearson's residuals-adjusted/standardized
        import numpy as np    
        import statsmodels.api as sm   
        table_nospliceregion = sm.stats.Table(df_gl_s_allgte5_nospliceregion)   # Standardized residuals

        ##print outputs to a dataframe
        #adjusted p-value
        if (p_nospliceregion*41>1):
            adjustedp_nospliceregion=1
        else:
            adjustedp_nospliceregion=p_nospliceregion*41
        
        #Pearson_residuals to output
        Pearson_residuals_standardized_nospliceregion = table_nospliceregion.standardized_resids
        P_R_S_gl_nospliceregion=defaultdict(list)
        for i in Pearson_residuals_standardized_nospliceregion.index:
            category_name_nospliceregion=f"{str(i)}"
            category_value_nospliceregion=Pearson_residuals_standardized_nospliceregion.loc[category_name_nospliceregion]['VEP_germline']
            P_R_S_gl_nospliceregion[category_name_nospliceregion].append(category_value_nospliceregion)

        
        #output df per gene
        output_df_nospliceregion = pd.DataFrame({'Gene': [folder_name], 'FLAG1':[FLAG1],'FLAG5':[FLAG5],'FLAGFISHER':[FLAGFISHER],'cbio starting len':[cbioCount1], 'cbio no intron etc': [cbioCount2], 'cbio no hypermutators':[len(cbio_VEP_nohypermutator)],'cbio no VUSLBB':[len(cbio_VEP_noVUSLBB)], 'chi-square': [c_nospliceregion], 'dof': [dof_nospliceregion], 'p-value': [p_nospliceregion], 'adjusted_p-value': [adjustedp_nospliceregion], 'number_germline_variants': [germline_counts_nospliceregion], 'number_germline_variants_inChiDF': [germline_counts_nospliceregion_1], 'number_somatic_variants': [somatic_counts_nospliceregion], 'number_somatic_variants_inChiDF': [somatic_counts_nospliceregion_1], 'frameshift_variant': [P_R_S_gl_nospliceregion['frameshift_variant']], 'stop_gained': [P_R_S_gl_nospliceregion['stop_gained']], 'splice_acceptor': [P_R_S_gl_nospliceregion['splice_acceptor_variant']], 'splice_donor': [P_R_S_gl_nospliceregion['splice_donor_variant']], 'missense_variant': [P_R_S_gl_nospliceregion['missense_variant']], 'gte_50bp_indel': [P_R_S_gl_nospliceregion['>=50bp_indel']], 'inframe_deletion': [P_R_S_gl_nospliceregion['inframe_deletion']], 'inframe_insertion': [P_R_S_gl_nospliceregion['inframe_insertion']],'start_lost': [P_R_S_gl_nospliceregion['start_lost']],'stop_lost': [P_R_S_gl_nospliceregion['stop_lost']],'coding_sequence_variant': [P_R_S_gl_nospliceregion['coding_sequence_variant']],'protein_altering_variant': [P_R_S_gl_nospliceregion['protein_altering_variant']],'#cbiovars_preVUSLBBfilter':[len(cbio_VEP)],'#cbiovars_after_VUSLBBfilter':[len(cbio_VEP_noVUSLBB)],"%vars_lost_to_VUSLBBfilter":[((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP)]})
        allTSG_df_nospliceregion = pd.concat([allTSG_df_nospliceregion,output_df_nospliceregion]) 
        
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

Fri Aug 23 14:59:26 2024
shared count
22
shared that are VUSLBB
7
cbio VEP len
213
cbio VEP len after filter VEP categories
192
with hypermutators count QC same as above
192
without hypermutators count
133
% vars lost to hypermutator filter=#total-#afterfilter/#total
30.729166666666668
without VUSLBB
131
# variants lost after VUSLBB filter
2
% vars lost to VUSLBB filter=#total-#afterfilter/#total
1.5037593984962405
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           3.0          6.0
splice_donor_variant             11.0          9.4
stop_gained                      32.0         32.6
frameshift_variant               30.0         52.8
missense_variant                 63.0         14.2
36.31348310498708 2.494301653018657e-07 4
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           3.0          6.0
splice_donor_variant         

shared count
9
shared that are VUSLBB
5
cbio VEP len
172
cbio VEP len after filter VEP categories
153
with hypermutators count QC same as above
153
without hypermutators count
47
% vars lost to hypermutator filter=#total-#afterfilter/#total
69.28104575163398
without VUSLBB
43
# variants lost after VUSLBB filter
4
% vars lost to VUSLBB filter=#total-#afterfilter/#total
8.51063829787234
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           1.0          8.0
splice_donor_variant              0.0          1.0
stop_gained                      10.0         21.0
frameshift_variant                7.0          8.0
start_lost                        1.0          1.0
inframe_insertion                 1.0          0.0
inframe_deletion                  5.0          0.0
missense_variant                116.0          4.0
95.31065319901403 2.3821484751796903e-18 6
                         VEP_germline  VEP_somatic
0       

/var/folders/5h/8jz2ndj504sb9gtp0mt16p680000gn/T/ipykernel_13862/2819388091.py:42: DtypeWarning: Columns (44,48) have mixed types. Specify dtype option on import or set low_memory=False.
  cbio_VEP=pd.read_csv("Python_coderun_7-11-24_cbio_process_again/Python_cbio_processed_VEP_MANE_CAID_8-16-24.txt", sep="\t")


cbio VEP len
23269
cbio VEP len after filter VEP categories
23034
with hypermutators count QC same as above
23034
without hypermutators count
19781
% vars lost to hypermutator filter=#total-#afterfilter/#total
14.12260137188504
without VUSLBB
17785
# variants lost after VUSLBB filter
1996
% vars lost to VUSLBB filter=#total-#afterfilter/#total
10.09049087508215
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant          102.0        695.2
splice_donor_variant             105.0        624.4
stop_gained                      192.0       2502.6
frameshift_variant               462.0       2398.6
start_lost                         0.0          1.0
inframe_insertion                  3.0         70.0
inframe_deletion                  47.0        322.8
missense_variant                1008.0      10216.0
protein_altering_variant           1.0         16.8
>=50bp_indel                      10.0         22.6
178.03660783

shared count
71
shared that are VUSLBB
26
cbio VEP len
847
cbio VEP len after filter VEP categories
806
with hypermutators count QC same as above
806
without hypermutators count
520
% vars lost to hypermutator filter=#total-#afterfilter/#total
35.483870967741936
without VUSLBB
457
# variants lost after VUSLBB filter
63
% vars lost to VUSLBB filter=#total-#afterfilter/#total
12.115384615384615
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           18.0         29.0
splice_donor_variant              25.0         24.2
stop_gained                       87.0        121.8
frameshift_variant                85.0         92.0
inframe_deletion                   0.0          5.0
missense_variant                  87.0        163.0
protein_altering_variant           1.0          0.0
>=50bp_indel                       4.0          1.0
16.292132003540548 0.01226897094007006 6
                         VEP_germline  VEP_

...

shared count
75
shared that are VUSLBB
4
cbio VEP len
314
cbio VEP len after filter VEP categories
301
with hypermutators count QC same as above
301
without hypermutators count
230
% vars lost to hypermutator filter=#total-#afterfilter/#total
23.588039867109636
without VUSLBB
229
# variants lost after VUSLBB filter
1
% vars lost to VUSLBB filter=#total-#afterfilter/#total
0.43478260869565216
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           41.0         12.0
splice_donor_variant              53.0         12.0
stop_gained                      183.0         57.0
frameshift_variant               349.0        140.0
stop_lost                          1.0          0.0
start_lost                        23.0          0.0
inframe_insertion                  1.0          0.0
inframe_deletion                   9.0          0.0
missense_variant                 218.0          2.0
protein_altering_variant         

shared count
81
shared that are VUSLBB
27
cbio VEP len
324
cbio VEP len after filter VEP categories
280
with hypermutators count QC same as above
280
without hypermutators count
152
% vars lost to hypermutator filter=#total-#afterfilter/#total
45.714285714285715
without VUSLBB
138
# variants lost after VUSLBB filter
14
% vars lost to VUSLBB filter=#total-#afterfilter/#total
9.210526315789474
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           52.0         13.6
splice_donor_variant              77.0          5.0
stop_gained                      287.0         38.8
frameshift_variant               505.0         34.6
inframe_insertion                  4.0          0.0
inframe_deletion                   1.0          0.0
missense_variant                  56.0         34.0
protein_altering_variant           3.0          0.0
>=50bp_indel                       7.0          1.0
83.16210308949479 3.7221609160901

shared count
119
shared that are VUSLBB
55
cbio VEP len
1951
cbio VEP len after filter VEP categories
1934
with hypermutators count QC same as above
1934
without hypermutators count
1453
% vars lost to hypermutator filter=#total-#afterfilter/#total
24.87073422957601
without VUSLBB
1209
# variants lost after VUSLBB filter
244
% vars lost to VUSLBB filter=#total-#afterfilter/#total
16.792842395044737
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           10.0         67.4
splice_donor_variant               7.0         17.0
stop_gained                       41.0        447.0
frameshift_variant                77.0        431.8
stop_lost                          0.0          1.0
start_lost                         0.0          1.0
inframe_insertion                 18.0          1.0
inframe_deletion                   3.0         32.4
missense_variant                 168.0        176.4
protein_altering_variant  

shared count
282
shared that are VUSLBB
27
cbio VEP len
720
cbio VEP len after filter VEP categories
703
with hypermutators count QC same as above
703
without hypermutators count
391
% vars lost to hypermutator filter=#total-#afterfilter/#total
44.38122332859175
without VUSLBB
370
# variants lost after VUSLBB filter
21
% vars lost to VUSLBB filter=#total-#afterfilter/#total
5.370843989769821
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant          466.0         15.0
splice_donor_variant             493.0         12.4
stop_gained                     3995.0        121.6
frameshift_variant             11095.0        199.0
start_lost                        36.0          0.0
inframe_insertion                 11.0          0.0
inframe_deletion                  16.0          0.0
missense_variant                 712.0         13.0
protein_altering_variant           1.0          0.0
>=50bp_indel                     

shared count
89
shared that are VUSLBB
18
cbio VEP len
678
cbio VEP len after filter VEP categories
655
with hypermutators count QC same as above
655
without hypermutators count
583
% vars lost to hypermutator filter=#total-#afterfilter/#total
10.992366412213741
without VUSLBB
574
# variants lost after VUSLBB filter
9
% vars lost to VUSLBB filter=#total-#afterfilter/#total
1.5437392795883362
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          48.0         52.2
splice_donor_variant             37.0         41.0
stop_gained                      88.0        154.6
frameshift_variant              207.0        274.8
stop_lost                         0.0          2.0
start_lost                        4.0          7.0
inframe_deletion                  0.0          7.0
missense_variant                 13.0         19.0
>=50bp_indel                      2.0          3.4
11.049459446719968 0.08685980356394409 6
   

## COSMIC vs ClinVar:

modified code from above here for COSMIC (did not change variable names so cbio here means cosmic)

In [19]:
start_time_block2=time.asctime(time.localtime())
print(start_time_block2)
#initialize dfs to collect info gene-wise
#1-VEP categories that are filtered out (expected to be filtered by cbio so removing from analysis, but outputting to this df for counts and QC)
VEP_categories_filtered_45TSGs=pd.DataFrame()
#2-all TSG, including splice region, all output stats to this df gene-wise
allTSG_df=pd.DataFrame()
#3-all TSG, excluding splice region, all output stats to this df gene-wise
allTSG_df_nospliceregion=pd.DataFrame()

FLAG1_count=FLAG5_count=FLAGFISHER_count=0

parentpath="USER INSERT PARENT PATH TO WHERE OTHER USER INPUT FILES REQUIRED BELOW ARE STORED"
os.chdir(parentpath)
# loading VEP consequence titles (category names) to use as index to join cbio and clinvar processed dfs:
VEP_calc_consequences=pd.read_csv("USER INPUT FILE WITH LIST OF POSSIBLE CONSEQUENCES FROM ENSEMBL VEP", sep="\t", header=None)


#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE TRANSCRIPT ID (Supplemental Table S4)", sep="\t")


parentpath2="USER INPUT PATH TO DATA FROM COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38"
os.chdir(parentpath2)

for index, row in TranscriptID.iterrows():
    
    # Loop through each folder name
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        clinvar_cbio_tocompare=pd.read_csv("USER INPUT FILE NAME OF SOMATIC VARIANTS IN CLINVAR UNFILTERED FOR PATHOGENICITY", sep="\t")
        Variation_interpretation_toexclude = ["Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic", "Conflicting interpretations of pathogenicity greater than or equal to 75"]
        clinvar_cbio_tocompare_VUSLBB=clinvar_cbio_tocompare[~clinvar_cbio_tocompare["GL_first_Description"].isin(Variation_interpretation_toexclude)]
        print("shared count")
        print(len(clinvar_cbio_tocompare))
        print("shared that are VUSLBB")
        print(len(clinvar_cbio_tocompare_VUSLBB))
        cbio_VEP=pd.read_csv("USER INPUT PROCESSED SOMATIC DATA FILE NAME", sep="\t")
        print("cbio VEP len")
        cbioCount1=len(cbio_VEP)
        print(len(cbio_VEP))
        VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant", "splice_region_variant"] #8-23-24 #, ">=50bp_indel"]
        cbio_VEP=cbio_VEP[~cbio_VEP["collapsed_Consequence"].isin(VEP_categories_tofilter_list)]
        print("cbio VEP len after filter VEP categories")
        print(len(cbio_VEP))
        cbioCount2=len(cbio_VEP)
        removetheseVUSLBB=clinvar_cbio_tocompare_VUSLBB["@id"]
        cbio_VEP_noVUSLBB=cbio_VEP[~cbio_VEP["@id"].isin(removetheseVUSLBB)]
        print("without VUSLBB")
        print(len(cbio_VEP_noVUSLBB))
        print("# variants lost after VUSLBB filter")
        print(len(cbio_VEP)-len(cbio_VEP_noVUSLBB))
        print("% vars lost to VUSLBB filter=#total-#afterfilter/#total")
        print(((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP))
        cbio_VEP_cancertype=cbio_VEP_noVUSLBB
        #initialize default dict to select 1 variant at random from list of variants per patient (for patients with multiple variants in the same gene):
        cbio_VEP_dict=defaultdict(list)
        # loading VEP consequence titles (categories) to use as index to join randomavg dfs:
        cbio_VEP_calc_consequences=VEP_calc_consequences
        cbio_VEP_randomavg_means=cbio_VEP_calc_consequences.set_index(0)
        #making dictionary of list of variants per patient id in the form ({patient1: [var1consequence, var2consequence, var3consequence,...], patient2:[var1consequence,var2consequence,..]})
        for index, row in cbio_VEP_cancertype.iterrows():
            patientid=cbio_VEP_cancertype.loc[index]["INDIVIDUAL_ID"]
            VEPresult=cbio_VEP_cancertype.loc[index]["collapsed_Consequence"]
            cbio_VEP_dict[patientid].append(VEPresult)

        
        #randomly selecting 1 variant per patient id and summarizing distribution, repeating 5 times
        for i in range(5):
            cbio_VEP_random = {k:random.choice(v) for k,v in list(cbio_VEP_dict.items())}
            list_VEP_random=[]
            for key, val in cbio_VEP_random.items():
                eachrow=[key,val]
                list_VEP_random.append(eachrow)
            #convert list to df of 1 variant per patient [patient1:var2consequence, patient2:var1consequence, ...]
            dfconvert= pd.DataFrame(list_VEP_random)
            #summarize counts per VEP category and convert to df having VEP category as index
            cbio_VEP_randomavg=pd.DataFrame(dfconvert[1].value_counts())
            #above is a df of current random selection having row header=VEP category and only 1 column of the counts per category (aka row) derived from current random selection
            #rename counts column in above df with the loop number (1/2/3/4/5) + join together with previous df (if not the 1st random selection of 5) and calc mean of all 5 sets of distributions
            rename=f"VEP_count_{i}"
            cbio_VEP_randomavg_rename=cbio_VEP_randomavg.rename(columns={1:rename})
            cbio_VEP_randomavg_means=cbio_VEP_randomavg_means.join(cbio_VEP_randomavg_rename)
        #replace all missing values with 0 to calc mean
        cbio_VEP_randomavg_means_fill0=cbio_VEP_randomavg_means.fillna(0)
        #calc mean
        cbio_VEP_randomavg_means_fill0['mean']=cbio_VEP_randomavg_means_fill0.mean(axis=1)
        
        
        
        os.chdir(parentpath)
        os.chdir(folder_name)
        
        clinvar_VEP_cancertype=pd.read_csv("USER INPUT PROCESSED GERMLINE DATA FILE NAME", sep="\t")
        
        clinvar_VEP_MANE=VEP_calc_consequences.set_index(0)
        #create list with each VEP category in data, plus occurrence list of number of times each variant seen in that category
        clinvar_VEPresult=defaultdict(list)
        for index, row in clinvar_VEP_cancertype.iterrows():
            VEPresult=clinvar_VEP_cancertype.loc[index]["collapsed_Consequence"]
            variantcount=clinvar_VEP_cancertype.loc[index]["Occurrence"]
            clinvar_VEPresult[VEPresult].append(variantcount)
        #sum the list of occurrence per VEP category
        clinvar_VEPresult_sumoccurrence = {k: sum(v) for (k, v) in clinvar_VEPresult.items()}
        #create dataframe from dictionary
        clinvar_dfconvert=pd.DataFrame.from_dict(clinvar_VEPresult_sumoccurrence, orient='index')
        #join with VEP consequence df
        clinvar_VEP=clinvar_VEP_MANE.join(clinvar_dfconvert)
        #replace missing values with 0
        clinvar_VEP_fill0=clinvar_VEP.fillna(0)
        #rename germline column
        clinvar_VEP_fill0_rename=clinvar_VEP_fill0.rename(columns={0:"VEP_germline"})
        #rename somatic column in cbio df
        cbio_VEP_randomavg_means_fill0_rename=cbio_VEP_randomavg_means_fill0.rename(columns={"mean":"VEP_somatic"})
        #combine gl and s dfs
        df_gl_s = clinvar_VEP_fill0_rename.join(cbio_VEP_randomavg_means_fill0_rename)
        #filter out random selection columns
        df_gl_s=df_gl_s.filter(items=["VEP_germline","VEP_somatic"])
        #filter out categories filtered by cbio (excep for TERT- diff set since cbio includes promoter variants)
        if folder_name=="TERT":
            VEP_categories_tofilter_list=["synonymous_variant", "3_prime_UTR_variant", "intron_variant", "downstream_gene_variant"]
        else:    
            VEP_categories_tofilter_list=["synonymous_variant","5_prime_UTR_variant", "3_prime_UTR_variant", "non_coding_transcript_exon_variant", "intron_variant", "non_coding_transcript_variant", "upstream_gene_variant", "downstream_gene_variant", "regulatory_region_ablation", "regulatory_region_amplification", "regulatory_region_variant", "intergenic_variant"]

        df_gl_s_filter=df_gl_s.drop(VEP_categories_tofilter_list)
        
        #create df of VEP cat filtered out for all 45 TSGs (to QC)
        VEP_categories_filtered=df_gl_s.drop(df_gl_s_filter.index)
        #VEP_categories_filtered.to_csv("Python_coderun_8-24-23/Python_VEP_categories_filtered_8-29-23.txt", sep="\t")
        Gene= pd.DataFrame({0: [folder_name]})
        Germline= pd.DataFrame({0: ["Germline"]})
        Somatic= pd.DataFrame({0: ["Somatic"]})
        VEP_categories_filtered_output=pd.concat([Gene, Germline, VEP_categories_filtered["VEP_germline"], Somatic, VEP_categories_filtered["VEP_somatic"]])
        VEP_categories_filtered_45TSGs=pd.concat([VEP_categories_filtered_45TSGs,VEP_categories_filtered_output])
        
        df_gl_s_allgte5 = df_gl_s_filter[(df_gl_s_filter["VEP_somatic"] >0)| (df_gl_s_filter["VEP_germline"] >0)]
        
        
        #filter out splice region for sep chi square stats
        if "splice_region_variant" in df_gl_s_allgte5.index:
            df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5.drop("splice_region_variant")
        else:
            df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5
        
        
        
        #REPEAT WITHOUT SPLICE REGION VARIANTS
        #get total counts of gl and s variants analyzed
        #gl and s total counts
        somatic_counts_nospliceregion=df_gl_s_allgte5_nospliceregion['VEP_somatic'].sum()
        germline_counts_nospliceregion=df_gl_s_allgte5_nospliceregion['VEP_germline'].sum()
        totalcategories=len(df_gl_s_allgte5_nospliceregion)
        print(df_gl_s_allgte5_nospliceregion)
        for index, row in df_gl_s_allgte5_nospliceregion.iterrows():
            if (df_gl_s_allgte5_nospliceregion.loc[index,"VEP_germline"]<(0.01*germline_counts_nospliceregion))&(df_gl_s_allgte5_nospliceregion.loc[index,"VEP_somatic"]<(0.01*somatic_counts_nospliceregion)):
                df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5_nospliceregion.drop(index)
        
        #run chi-square test from scipy
        from scipy.stats import chi2_contingency
        c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5_nospliceregion)
        print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
        print(df_gl_s_allgte5_nospliceregion)
        print(expected_nospliceregion)

        expected_df=pd.DataFrame(expected_nospliceregion)

        #codeblock for rule exp<5 in both gl and s:
        df_gl_s_allgte5_nospliceregion_resetindex=df_gl_s_allgte5_nospliceregion.reset_index()

        indextodrop=[]
        for index, row in expected_df.iterrows():
            if ((expected_df.loc[index,0]<5)&(expected_df.loc[index,1]<5)):
                indextodrop.append(index)
        df_gl_s_allgte5_nospliceregion_resetindex_filter=df_gl_s_allgte5_nospliceregion_resetindex.drop(indextodrop)
        expected_df=expected_df.drop(indextodrop)
        
        df_gl_s_allgte5_nospliceregion=df_gl_s_allgte5_nospliceregion_resetindex_filter.set_index(0)
        
        c_nospliceregion, p_nospliceregion, dof_nospliceregion, expected_nospliceregion = chi2_contingency(df_gl_s_allgte5_nospliceregion)
        print(c_nospliceregion, p_nospliceregion, dof_nospliceregion) 
        print(df_gl_s_allgte5_nospliceregion)
        print(expected_nospliceregion)

        expected_df=pd.DataFrame(expected_nospliceregion)
        
        somatic_counts_nospliceregion_1=df_gl_s_allgte5_nospliceregion['VEP_somatic'].sum()
        germline_counts_nospliceregion_1=df_gl_s_allgte5_nospliceregion['VEP_germline'].sum()
        
        
        #FLAGS:
        FLAG1=FLAG5=FLAGFISHER=0

        for index, row in expected_df.iterrows():
            if (expected_df.loc[index,0]<1)|(expected_df.loc[index,1]<1):
                FLAG1=1
                FLAG1_count=FLAG1_count+1
       
        
        lt5cellcount=pd.DataFrame(expected_df[expected_df<5].count())
        lt5cellcount=lt5cellcount[0].sum()
        if lt5cellcount>(0.2*expected_df.size):
            FLAG5=1
            FLAG5_count=FLAG5_count+1
        
        for index, row in df_gl_s_allgte5_nospliceregion.iterrows():
            if (df_gl_s_allgte5_nospliceregion.loc[index,"VEP_germline"]<1)|(df_gl_s_allgte5_nospliceregion.loc[index,"VEP_somatic"]<1):
                FLAGFISHER=1
                FLAGFISHER_count=FLAGFISHER_count+1
        
        print("FLAG1")
        print(FLAG1)
        print("FLAG5")
        print(FLAG5)
        print("FLAGFISHER")
        print(FLAGFISHER)
        
        #get Pearson's residuals-adjusted/standardized
        import numpy as np    
        import statsmodels.api as sm   
        table_nospliceregion = sm.stats.Table(df_gl_s_allgte5_nospliceregion)   # Standardized residuals

        ##print outputs to a dataframe
        #adjusted p-value
        if (p_nospliceregion*41>1):
            adjustedp_nospliceregion=1
        else:
            adjustedp_nospliceregion=p_nospliceregion*41
        
        #Pearson_residuals to output
        Pearson_residuals_standardized_nospliceregion = table_nospliceregion.standardized_resids
        P_R_S_gl_nospliceregion=defaultdict(list)
        for i in Pearson_residuals_standardized_nospliceregion.index:
            category_name_nospliceregion=f"{str(i)}"
            category_value_nospliceregion=Pearson_residuals_standardized_nospliceregion.loc[category_name_nospliceregion]['VEP_germline']
            P_R_S_gl_nospliceregion[category_name_nospliceregion].append(category_value_nospliceregion)

        
        #output df per gene
        output_df_nospliceregion = pd.DataFrame({'Gene': [folder_name], 'FLAG1':[FLAG1],'FLAG5':[FLAG5],'FLAGFISHER':[FLAGFISHER],'cbio starting len':[cbioCount1], 'cbio no intron etc': [cbioCount2],'cbio no VUSLBB':[len(cbio_VEP_noVUSLBB)], 'chi-square': [c_nospliceregion], 'dof': [dof_nospliceregion], 'p-value': [p_nospliceregion], 'adjusted_p-value': [adjustedp_nospliceregion], 'number_germline_variants': [germline_counts_nospliceregion], 'number_germline_variants_inChiDF': [germline_counts_nospliceregion_1], 'number_somatic_variants': [somatic_counts_nospliceregion], 'number_somatic_variants_inChiDF': [somatic_counts_nospliceregion_1], 'frameshift_variant': [P_R_S_gl_nospliceregion['frameshift_variant']], 'stop_gained': [P_R_S_gl_nospliceregion['stop_gained']], 'splice_acceptor': [P_R_S_gl_nospliceregion['splice_acceptor_variant']], 'splice_donor': [P_R_S_gl_nospliceregion['splice_donor_variant']], 'missense_variant': [P_R_S_gl_nospliceregion['missense_variant']], 'gte_50bp_indel': [P_R_S_gl_nospliceregion['>=50bp_indel']], 'inframe_deletion': [P_R_S_gl_nospliceregion['inframe_deletion']], 'inframe_insertion': [P_R_S_gl_nospliceregion['inframe_insertion']],'start_lost': [P_R_S_gl_nospliceregion['start_lost']],'stop_lost': [P_R_S_gl_nospliceregion['stop_lost']],'coding_sequence_variant': [P_R_S_gl_nospliceregion['coding_sequence_variant']],'protein_altering_variant': [P_R_S_gl_nospliceregion['protein_altering_variant']],'#cbiovars_preVUSLBBfilter':[len(cbio_VEP)],'#cbiovars_after_VUSLBBfilter':[len(cbio_VEP_noVUSLBB)],"%vars_lost_to_VUSLBBfilter":[((len(cbio_VEP)-len(cbio_VEP_noVUSLBB))*100)/len(cbio_VEP)]})
        allTSG_df_nospliceregion = pd.concat([allTSG_df_nospliceregion,output_df_nospliceregion]) 
        
        
        
        print("Current directory:", os.getcwd())
        os.chdir(parentpath2)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time_block2=time.asctime(time.localtime())
print(end_time_block2)

Fri Sep  6 19:14:51 2024
shared count
25
shared that are VUSLBB
9
cbio VEP len
825
cbio VEP len after filter VEP categories
806
without VUSLBB
802
# variants lost after VUSLBB filter
4
% vars lost to VUSLBB filter=#total-#afterfilter/#total
0.49627791563275436
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           3.0         20.6
splice_donor_variant             11.0          8.2
stop_gained                      32.0        142.8
frameshift_variant               30.0        522.2
stop_lost                         0.0          1.0
inframe_insertion                 0.0          7.0
missense_variant                 63.0          2.0
>=50bp_indel                      0.0          6.2
375.32486668128513 5.9556089134040215e-80 4
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           3.0         20.6
splice_donor_variant            

shared count
6
shared that are VUSLBB
5
cbio VEP len
67
cbio VEP len after filter VEP categories
56
without VUSLBB
45
# variants lost after VUSLBB filter
11
% vars lost to VUSLBB filter=#total-#afterfilter/#total
19.642857142857142
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           1.0          4.6
splice_donor_variant              0.0          2.0
stop_gained                      10.0         17.2
frameshift_variant                7.0         13.4
start_lost                        1.0          0.0
inframe_insertion                 1.0          0.0
inframe_deletion                  5.0          0.0
missense_variant                116.0          5.8
86.30889980218289 4.001003905610515e-17 5
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant           1.0          4.6
splice_donor_variant              0.0          2.0
stop_gained 

/var/folders/5h/8jz2ndj504sb9gtp0mt16p680000gn/T/ipykernel_10005/3892583910.py:48: DtypeWarning: Columns (48,52) have mixed types. Specify dtype option on import or set low_memory=False.
  cbio_VEP=pd.read_csv("Extract_10-2-23/COSMIC_processed_VEP_MANE_CAID_9-6-24.txt", sep="\t")


cbio VEP len
29582
cbio VEP len after filter VEP categories
29404
without VUSLBB
25647
# variants lost after VUSLBB filter
3757
% vars lost to VUSLBB filter=#total-#afterfilter/#total
12.77717317371786
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant          102.0        621.6
splice_donor_variant             105.0        520.8
stop_gained                      192.0       3193.4
frameshift_variant               462.0       2229.8
inframe_insertion                  3.0         32.4
inframe_deletion                  47.0        411.0
missense_variant                1008.0      17155.0
protein_altering_variant           1.0         14.0
>=50bp_indel                      10.0         25.0
616.4174616740237 5.733133400090097e-131 5
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant         102.0        621.6
splice_donor_variant          

shared count
69
shared that are VUSLBB
25
cbio VEP len
472
cbio VEP len after filter VEP categories
445
without VUSLBB
400
# variants lost after VUSLBB filter
45
% vars lost to VUSLBB filter=#total-#afterfilter/#total
10.112359550561798
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           18.0         21.2
splice_donor_variant              25.0         34.2
stop_gained                       87.0        110.6
frameshift_variant                85.0        102.0
inframe_deletion                   0.0          2.0
missense_variant                  87.0         92.8
protein_altering_variant           1.0          0.0
>=50bp_indel                       4.0          2.2
1.9221610554671615 0.8598073286264066 5
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          18.0         21.2
splice_donor_variant             25.0         34.2

...

shared count
62
shared that are VUSLBB
1
cbio VEP len
307
cbio VEP len after filter VEP categories
305
without VUSLBB
305
# variants lost after VUSLBB filter
0
% vars lost to VUSLBB filter=#total-#afterfilter/#total
0.0
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           41.0          8.2
splice_donor_variant              53.0         15.0
stop_gained                      183.0         75.6
frameshift_variant               349.0        190.0
stop_lost                          1.0          0.0
start_lost                        23.0          0.0
inframe_insertion                  1.0          0.0
inframe_deletion                   9.0          0.0
missense_variant                 218.0          4.2
protein_altering_variant           1.0          0.0
>=50bp_indel                       6.0          2.0
108.58214007661077 4.03747547558672e-21 6
                         VEP_germline  VEP_somatic
0          

shared count
57
shared that are VUSLBB
16
cbio VEP len
429
cbio VEP len after filter VEP categories
419
without VUSLBB
347
# variants lost after VUSLBB filter
72
% vars lost to VUSLBB filter=#total-#afterfilter/#total
17.183770883054894
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant           52.0          1.0
splice_donor_variant              77.0          5.0
stop_gained                      287.0         41.8
frameshift_variant               505.0         34.6
start_lost                         0.0          2.0
inframe_insertion                  4.0          0.0
inframe_deletion                   1.0          0.0
missense_variant                  56.0        230.6
protein_altering_variant           3.0          0.0
>=50bp_indel                       7.0          0.0
639.155216405903 5.1896898257107595e-137 4
                         VEP_germline  VEP_somatic
0                                            

cbio VEP len
1760
cbio VEP len after filter VEP categories
1749
without VUSLBB
1477
# variants lost after VUSLBB filter
272
% vars lost to VUSLBB filter=#total-#afterfilter/#total
15.551743853630645
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          10.0         98.0
splice_donor_variant              7.0         33.2
stop_gained                      41.0        652.0
frameshift_variant               77.0        366.6
stop_lost                         0.0          1.0
inframe_insertion                18.0          1.0
inframe_deletion                  3.0         16.6
missense_variant                168.0        260.6
>=50bp_indel                      2.0          8.0
274.53905694865983 2.3176451005299676e-56 6
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          10.0         98.0
splice_donor_variant              7.0      

shared count
215
shared that are VUSLBB
28
cbio VEP len
444
cbio VEP len after filter VEP categories
419
without VUSLBB
391
# variants lost after VUSLBB filter
28
% vars lost to VUSLBB filter=#total-#afterfilter/#total
6.682577565632458
                          VEP_germline  VEP_somatic
0                                                  
splice_acceptor_variant          466.0         13.0
splice_donor_variant             493.0          4.0
stop_gained                     3995.0        103.4
frameshift_variant             11095.0        229.6
start_lost                        36.0          1.0
inframe_insertion                 11.0          0.0
inframe_deletion                  16.0          0.0
missense_variant                 712.0         22.0
protein_altering_variant           1.0          0.0
>=50bp_indel                      70.0          1.0
10.879703344546442 0.027949774016865575 4
                         VEP_germline  VEP_somatic
0                                             

shared count
88
shared that are VUSLBB
14
cbio VEP len
602
cbio VEP len after filter VEP categories
589
without VUSLBB
574
# variants lost after VUSLBB filter
15
% vars lost to VUSLBB filter=#total-#afterfilter/#total
2.5466893039049237
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          48.0         65.2
splice_donor_variant             37.0         32.0
stop_gained                      88.0        139.0
frameshift_variant              207.0        286.6
stop_lost                         0.0          1.0
start_lost                        4.0          7.0
inframe_deletion                  0.0          4.0
missense_variant                 13.0         15.2
>=50bp_indel                      2.0          6.0
6.096406050163269 0.41247779767069265 6
                         VEP_germline  VEP_somatic
0                                                 
splice_acceptor_variant          48.0         65.2
splice_do